<a href="https://colab.research.google.com/github/Namballa-95/Data-Science-project/blob/main/NLP_Task02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
#LOADIN THE DATASET
df = pd.read_csv("/content/USA_Housing.csv")  # Example: IMDb / Amazon / Twitter
df.head()

In [ ]:
# DATA UNDERSTANDING
print("Total Samples:", len(df))
print("\nClass Distribution:\n", df['sentiment'].value_counts())
# Sample text
print("\nSample Text:\n", df['text'][0])

In [ ]:
# NLP PREPROCESSING
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)
df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text']].head()

In [ ]:
# FEATURE ENGINEERING
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df['clean_text'])

In [ ]:
# TF __IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

In [ ]:
# TRAIN TEST SPLIT
y = df['sentiment']
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [ ]:
# MODEL TRAINING
# 1) LOGISTIC REGRESSION
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

# 2) NAIVE BAYES
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

# 3) DECISION TREE
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)
y_pred_dt = dt.predict(X_test_bow)

In [ ]:
# MODEL EVALUATION FUNCTION
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n📊 {model_name} Performance")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted'))
    print("Recall:", recall_score(y_true, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

In [ ]:
# MODEL EVLUATION
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")

# FINAL PIPELINE
Raw Text
   ↓
Preprocessing
   ↓
Feature Engineering (BoW / TF-IDF)
   ↓
Model Training
   ↓
Evaluation
   ↓
Comparison